# House Price Prediction — Modelling

This notebook builds and evaluates machine learning models to predict house prices in West Yorkshire using preprocessed Zoopla listings data.

**Models used:**
- Linear Regression (baseline)
- Random Forest Regressor
- XGBoost Regressor

**Steps:**
1. Load preprocessed data
2. Define features and target
3. Build sklearn Pipelines for each model
4. Evaluate and compare models
5. Visualise feature importances
6. Hyperparameter tuning with GridSearchCV
7. Final model comparison

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings('ignore')

print('All imports successful')

## 2. Load Preprocessed Data

The data was preprocessed in `01_preprocessing_and_eda.ipynb`. Load the cleaned CSV from the `data/` folder.

In [ ]:
df = pd.read_csv('../data/house_prices_west_yorkshire.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 3. Define Features and Target

In [ ]:
X = df.drop(columns='price')
y = df['price']

# Identify column types for preprocessing
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numerical_features = X.select_dtypes(exclude=['object']).columns.tolist()

print(f'Numerical features: {numerical_features}')
print(f'Categorical features: {categorical_features}')

## 4. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')

## 5. Build Preprocessing Pipeline

We use a `ColumnTransformer` to apply `StandardScaler` to numerical features and `OneHotEncoder` to categorical features. This is wrapped in each model's `Pipeline` to prevent data leakage.

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

## 6. Define Model Pipelines

Each model is wrapped in a `Pipeline` combining the preprocessor and the regressor. This ensures the same preprocessing is applied consistently during training and prediction.

In [ ]:
linreg_model = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('regressor', LinearRegression())
])

rf_model = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

xgb_model = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('regressor', XGBRegressor(random_state=42, objective='reg:squarederror'))
])

## 7. Evaluation Function

A reusable function that fits the model, generates predictions, and returns MAE, RMSE, and R².

In [ ]:
def evaluate(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)

    print(f'  MAE:  £{mae:,.2f}')
    print(f'  RMSE: £{rmse:,.2f}')
    print(f'  R²:   {r2:.4f}')

    return r2, rmse, mae

## 8. Baseline Model Evaluation

In [ ]:
print('Linear Regression:')
r2, rmse, mae = evaluate(linreg_model, X_train, X_test, y_train, y_test)

print('\nRandom Forest:')
r2_rf, rmse_rf, mae_rf = evaluate(rf_model, X_train, X_test, y_train, y_test)

print('\nXGBoost:')
r2_xgb, rmse_xgb, mae_xgb = evaluate(xgb_model, X_train, X_test, y_train, y_test)

## 9. Actual vs Predicted Plots

Visualising how well each model's predictions align with actual prices. Points close to the red diagonal line indicate accurate predictions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = [
    (linreg_model, 'Linear Regression'),
    (rf_model, 'Random Forest'),
    (xgb_model, 'XGBoost')
]

for ax, (model, name) in zip(axes, models):
    y_pred = model.predict(X_test)
    ax.scatter(y_test, y_pred, alpha=0.4, s=10)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    ax.set_xlabel('Actual Price')
    ax.set_ylabel('Predicted Price')
    ax.set_title(f'Actual vs Predicted\n{name}')

plt.tight_layout()
plt.show()

## 10. Feature Importances

Comparing which features drive predictions in Random Forest and XGBoost.

In [ ]:
# Get feature names after preprocessing
onehot_columns = list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features))
all_features = numerical_features + onehot_columns

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, (model, name) in zip(axes, [(rf_model, 'Random Forest'), (xgb_model, 'XGBoost')]):
    importances = model.named_steps['regressor'].feature_importances_
    imp_df = pd.DataFrame({'Feature': all_features, 'Importance': importances})
    imp_df = imp_df.sort_values(by='Importance', ascending=False).head(10)
    sns.barplot(data=imp_df, x='Importance', y='Feature', ax=ax)
    ax.set_title(f'{name} — Top 10 Feature Importances')

plt.tight_layout()
plt.show()

## 11. Hyperparameter Tuning

Using `GridSearchCV` with 5-fold cross-validation to find the best hyperparameters for Random Forest and XGBoost.

> **Note:** This step is computationally expensive. Results from the best parameters are shown below.

In [ ]:
# Random Forest tuning
param_grid_rf = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [10, 20, None],
    'regressor__min_samples_split': [2, 5],
}

grid_search_rf = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid_rf,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)
grid_search_rf.fit(X_train, y_train)
print('Best Random Forest params:', grid_search_rf.best_params_)

In [ ]:
# XGBoost tuning
param_grid_xgb = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [3, 6, 9],
    'regressor__learning_rate': [0.01, 0.1, 0.2],
    'regressor__subsample': [0.8, 1.0],
    'regressor__colsample_bytree': [0.8, 1.0]
}

grid_search_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid_xgb,
    scoring='neg_root_mean_squared_error',
    cv=5,
    verbose=1,
    n_jobs=-1
)
grid_search_xgb.fit(X_train, y_train)
print('Best XGBoost params:', grid_search_xgb.best_params_)

## 12. Retrain with Best Parameters

In [ ]:
rf_model_best = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        min_samples_split=2,
        random_state=42
    ))
])

xgb_model_best = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.2,
        subsample=1.0,
        colsample_bytree=0.8,
        random_state=42
    ))
])

print('Random Forest (Tuned):')
r2_rf_t, rmse_rf_t, mae_rf_t = evaluate(rf_model_best, X_train, X_test, y_train, y_test)

print('\nXGBoost (Tuned):')
r2_xgb_t, rmse_xgb_t, mae_xgb_t = evaluate(xgb_model_best, X_train, X_test, y_train, y_test)

## 13. Final Model Comparison

Comparing all models before and after tuning. Note that hyperparameter tuning did not improve performance in this case — both tuned models performed slightly worse than their baseline counterparts. This is a known outcome when GridSearchCV overfits to the cross-validation folds, particularly with limited data. The baseline XGBoost remains the best performing model.

In [ ]:
comparison = pd.DataFrame({
    'Model': [
        'Linear Regression',
        'Random Forest (Baseline)',
        'XGBoost (Baseline)',
        'Random Forest (Tuned)',
        'XGBoost (Tuned)'
    ],
    'R²': [r2, r2_rf, r2_xgb, r2_rf_t, r2_xgb_t],
    'RMSE (£)': [rmse, rmse_rf, rmse_xgb, rmse_rf_t, rmse_xgb_t],
    'MAE (£)':  [mae, mae_rf, mae_xgb, mae_rf_t, mae_xgb_t]
})

comparison['R²'] = comparison['R²'].round(4)
comparison['RMSE (£)'] = comparison['RMSE (£)'].apply(lambda x: f'£{x:,.0f}')
comparison['MAE (£)']  = comparison['MAE (£)'].apply(lambda x: f'£{x:,.0f}')

print(comparison.to_string(index=False))

## 14. XGBoost Feature Importances (Tuned Model)

In [ ]:
importances = xgb_model_best.named_steps['regressor'].feature_importances_
feature_names = xgb_model_best.named_steps['preprocessing'].get_feature_names_out()
sorted_idx = importances.argsort()[::-1]

plt.figure(figsize=(10, 6))
sns.barplot(x=importances[sorted_idx][:10], y=feature_names[sorted_idx][:10])
plt.title('Top 10 XGBoost Feature Importances (Tuned Model)')
plt.tight_layout()
plt.show()

## Summary

- **Best model:** XGBoost (Baseline) — R² = 0.67, RMSE = £112,749, MAE = £68,932
- **Key predictors:** number of rooms, property group, borough, and distance to amenities
- **Hyperparameter tuning** did not improve performance — likely due to overfitting during cross-validation on this dataset size
- **Next steps:** Feature engineering, ensemble stacking, or a Streamlit deployment could further improve results